# Week 1: SurRoL setup and scripted demo

Installs SurRoL, runs scripted policies, saves a GIF and logs trajectories to DuckDB in Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/Tejas-1701/surgical-skill-bridge.git'
!test -d /content/surgical-skill-bridge || git clone -q $REPO_URL /content/surgical-skill-bridge
%cd /content/surgical-skill-bridge
!bash setup.sh

import sys
sys.path.insert(0, '/content/surgical-skill-bridge/src')

In [ ]:
!python scripts/run_scripted_demo.py --task needle_reach --episodes 5 --gif
!python scripts/run_scripted_demo.py --task peg_transfer --episodes 5 --gif

In [ ]:
from IPython.display import Image, display
from skillbridge.config import gif_dir

for gif_path in sorted(gif_dir().glob('*.gif')):
    print(gif_path.name)
    display(Image(filename=str(gif_path)))

## Write your own SQL

Tables: `episodes` and `timesteps`. Saved queries live in `queries/`.

In [ ]:
import duckdb
from skillbridge.config import database_path

connection = duckdb.connect(str(database_path()))
%load_ext sql
%sql connection

In [ ]:
%%sql
SELECT task, policy_type, COUNT(*) AS episodes, AVG(success::INTEGER) AS success_rate
FROM episodes
GROUP BY task, policy_type

In [ ]:
%%sql
SELECT * FROM timesteps WHERE episode_id = 1 ORDER BY t LIMIT 10

In [ ]:
connection.close()